# Preprocessing of data

# mRNA concentration in the cell

03.09.2025. Cedric Meyer. Highest expressed mRNAs during exponential growth phase are needed as a background transcriptome. Here I take data from Lu et al. 2007 who themselves average over three data sets with the argument that this can average out measurement errors.

### Convert copy number per cell into concentration

In [10]:
# packages
include("packages.jl")

# data 
data 		= to_DataFrame(readmatrix("../data/input/mRNA_copy_numbers.xlsx", "mRNA_copy_number", "A1:E2472"));

# Umrechnung in µM
volume_cell 		= 7*1.8e-15 #1.8e-15 		# 7x inflation with heterologous protein production according to Basan et al., 2015	# L source https://bionumbers.hms.harvard.edu/bionumber.aspx?s=n&v=4&id=108830 - could be even more see Bertaux et al. 2020
N 					= 6.022e23
c_one_molecule 		= 1/(volume_cell*N)	# mol/L 
c_one_molecule 	   /= 1e-6 				# µM
data.mRNA_µM 	    = data.mRNA_molecules_per_cell * c_one_molecule

# Only gene name and mRNA concentration 
only_concentrations         = DataFrame()
only_concentrations.gene    = data.Gene       
only_concentrations.mRNA    = data.mRNA_µM;

# Add heterologous genes 
data_heterologous 		    = to_DataFrame(readmatrix("../data/input/mRNA_steady_state_heterologous_proteins.xlsx", "mRNA_concentration", "A1:B3"))
append!(only_concentrations, data_heterologous);


🔧 Julia started with 8 threads


### Are sequences in E. coli gene set?

In [12]:
# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa"
seqs_to_use 	    = (1,2323)   
sequence_path = joinpath(@__DIR__, "..", "data", "sequences")
seq_file = sequence_path * "\\" * sequence_file
seqs, seq_names = seqs_import_transcripts(seq_file, seqs_to_use) # Function that transforms DNA Sequence into mRNA, amino acids and codons

mRNA_in_gene_set = String.(intersect(only_concentrations.gene, collect(keys(seqs))));
df_out = filter(:gene => x -> x in mRNA_in_gene_set, only_concentrations);

✅ Successfully loaded 2323 sequences


### Save df to excel

In [13]:

# Write in excel
XLSX.openxlsx("../data/input/mRNA_concentrations.xlsx", mode="rw") do xf
    sheet = xf["mRNA_concentration_high_cell_v"]
	XLSX.writetable!(sheet, df_out, anchor_cell=XLSX.CellRef("A1"))
end

# Combine fasta files

01.12.2025

In [7]:
folder = "../data/validation/rahmen_2015"

open("Ecoli_K12_MG1655_geneset_with_rahmen_2015.fasta", "w") do out
    for file in readdir(folder, join=true)
        # endswith(file, ".fasta") || continue
        open(file, "r") do f
            write(out, read(f, String))
            write(out, "\n")   # kleine Sicherheit, falls kein newline am Ende
        end
    end
end
